# 97 — Best-of-N Sampling
## Inference-Time Scaling with a Process Reward Model
⏱ ~60 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/97-best-of-n-sampling/best_of_n_sampling_workbook.ipynb)

Instead of fine-tuning a model to get better answers, **Best-of-N sampling** asks the model to answer the same question N times and then picks the best response. The trick is *how you pick the best*: a naive approach looks only at the final answer, but a **Process Reward Model (PRM)** evaluates the quality of every intermediate reasoning step — which is much harder to game and produces more reliable rankings.

This workshop builds a LangGraph pipeline that:
1. Fans out N parallel chain-of-thought generation calls (high temperature → diversity)
2. Scores each candidate with an LLM judge acting as a PRM
3. Returns the highest-scoring chain as the final answer

**Why it matters:** Best-of-N is a form of *inference-time compute scaling* — you spend more tokens at inference instead of more GPU-hours at training. Cobbe et al. 2021 showed that with a strong verifier, Best-of-N can match or exceed models with 30× more parameters.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — Best-of-N, PRMs, and inference-time scaling |
| 2 | **Setup** — Dependencies, API key, imports |
| 3 | **State design** — BonState and GenState TypedDicts |
| 4 | **Generation** — Fan-out with `Send()` and diverse chain-of-thought |
| 5 | **Process Reward Model** — JUDGE_PROMPT and scoring logic |
| 6 | **Graph assembly** — Connecting dispatch → generate → score_all |
| 7 | **End-to-end run** — Rate problem and constraint logic problem |
| 8 | **Analysis** — Reading the ranked outputs and understanding the winner |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab (free tier works)
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- Packages: `langchain-openai`, `langgraph`, `python-dotenv`

### Key References
> **Cobbe et al. 2021** — *Training Verifiers to Solve Math Word Problems*
> arxiv.org/abs/2110.14168 — The paper that established PRM-guided Best-of-N as a practical inference-time scaling strategy.
>
> **Lightman et al. 2023** — *Let's Verify Step by Step*
> arxiv.org/abs/2305.20050 — Extends PRMs to general reasoning, showing per-step supervision is essential.
>
> LangGraph docs: https://langchain-ai.github.io/langgraph/

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "langchain-openai", "langgraph", "python-dotenv"],
        check=True,
    )
    print("Colab install complete.")
else:
    print("Local environment — skipping install (use requirements.txt).")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

---
## Part 1 — Concepts: Best-of-N, PRMs, and Inference-Time Scaling

### What is Best-of-N Sampling?

The core idea is simple: generate N candidate answers independently, score them, and return the best.

```
  Question
     │
     ├──► generate() ──► Chain A  ──► score 7/10
     ├──► generate() ──► Chain B  ──► score 4/10
     ├──► generate() ──► Chain C  ──► score 9/10  ← winner
     └──► generate() ──► Chain D  ──► score 6/10
                                          │
                                    best answer
```

**High temperature** during generation creates diverse chains. Two chains might reach the same answer through different reasoning paths — but only one path will hold up under scrutiny.

### Why Not Just Pick the Most Common Answer (Majority Voting)?

Majority voting (also called self-consistency) counts how often each final answer appears across N samples and picks the plurality. It works when:
- Answers are discrete (multiple-choice, exact numbers)
- The model is mostly right but occasionally drifts

It fails when:
- Wrong answers cluster (a systematic bias makes the model repeatedly wrong the same way)
- The reasoning process contains errors even when the answer is accidentally correct

**Process Reward Models fix both failure modes.**

### Process Reward Models (PRMs) vs Outcome Reward Models (ORMs)

| Feature | ORM (Outcome Reward) | PRM (Process Reward) |
|---------|---------------------|---------------------|
| What it scores | Final answer only | Each intermediate step |
| Can detect | Wrong final answer | Wrong reasoning even with correct answer |
| Training data | (answer, correct/wrong) | (step, correct/wrong) per step |
| Robustness | Fooled by lucky guesses | More robust — requires valid reasoning |
| This example uses | — | LLM-as-PRM via structured prompt |

Our PRM prompt scores three dimensions:
1. **Step clarity** — each step is labeled and easy to follow
2. **Logical rigor** — no skipped deductions or unjustified leaps
3. **Correct answer** — the final answer is right

### Inference-Time Scaling

Traditional scaling: more parameters → better performance (expensive to train, fixed at inference).

Inference-time scaling: more compute at inference → better performance (cheap to parallelize).

```
  Cobbe et al. 2021 finding:
  Best-of-100 + verifier  ≈  model with 30× more parameters
  at a fraction of the training cost
```

This is why inference-time compute (chain-of-thought, Best-of-N, MCTS, etc.) has become a major research frontier.

---
## Part 2 — Setup: Core Imports

We need:
- `langchain_openai` — the OpenAI chat model wrapper
- `langgraph` — the graph execution engine with `Send()` for fan-out
- Standard library: `json`, `operator`, `typing`

Two model instances with **different temperatures**:
- `_gen` at `temperature=0.8` — diversity in generation
- `_judge` at `temperature=0` — stable, deterministic scoring

In [ ]:
import json
import operator
from typing import Annotated
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.constants import Send

# High temperature → diverse reasoning chains
_gen   = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)
# Zero temperature → stable, consistent reward signal
_judge = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("Imports OK.")
print(f"Generator temperature: {_gen.temperature}")
print(f"Judge temperature:     {_judge.temperature}")

---
## Part 3 — State Design: BonState and GenState

LangGraph passes a **state dict** between nodes. The shape of that dict determines what information flows through the graph.

### BonState — the main graph state

```
BonState
  question  : str        — the user's question (read-only after initialization)
  candidates: list[dict] — accumulates results from all N generate() calls
  scored    : list[dict] — full list after judging (score + critique per candidate)
  best      : str        — winner's answer
```

**Key design decision:** `candidates` uses `Annotated[list[dict], operator.add]`.

This tells LangGraph to *concatenate* lists rather than replace them. Without this annotation, if three `generate()` nodes write to `candidates` concurrently, only the last write would survive. With `operator.add`, all N results are merged.

### GenState — the subgraph state

Each parallel `generate()` call receives only `{question: str}`. It doesn't need to see the other candidates — that isolation is what makes the chains independent.

In [ ]:
class BonState(TypedDict):
    question:   str
    candidates: Annotated[list[dict], operator.add]  # merge lists from N parallel nodes
    scored:     list[dict]   # full scored list, set once by score_all
    best:       str          # winning chain's answer


class GenState(TypedDict):
    question: str


# Demonstrate how operator.add works for state merging
print("operator.add merges lists, not replaces:")
print(operator.add(["chain_A"], ["chain_B"]))
print(operator.add(["chain_A", "chain_B"], ["chain_C"]))

---
## Part 4 — Generation: Fan-out with `Send()` and Chain-of-Thought

### The `dispatch` function — conditional edge that fans out

`dispatch` is used as a *conditional edge function* from `START`. Instead of routing to a single node, it returns a **list of `Send()` objects** — one per desired sample. LangGraph executes all of them in parallel.

```python
# Sending N=4 parallel messages to the "generate" node
[
    Send("generate", {"question": "..."}),
    Send("generate", {"question": "..."}),
    Send("generate", {"question": "..."}),
    Send("generate", {"question": "..."}),
]
```

Each `Send()` is an independent invocation. They run concurrently, writing their results to `candidates`. Because `candidates` uses `operator.add`, all four results are collected safely.

### The `generate` function — one chain-of-thought reasoning path

System prompt instructs the model to think step by step and end with `Answer: <answer>`. The answer is extracted from the final `Answer:` line, with a fallback to the last non-empty line if the model doesn't follow the format perfectly.

In [ ]:
N_SAMPLES = 4  # reasoning chains to generate per question


def dispatch(state: BonState) -> list:
    """Fan out N independent generate calls — same question, different temperature seeds."""
    return [Send("generate", {"question": state["question"]}) for _ in range(N_SAMPLES)]


def generate(state: GenState) -> dict:
    """Produce one chain-of-thought solution. High temp → diverse reasoning paths."""
    reply = _gen.invoke([
        {"role": "system", "content": "Solve step by step. End your response with 'Answer: <answer>'"},
        {"role": "user",   "content": state["question"]},
    ])
    text = reply.content.strip()
    # Extract declared answer; fall back to the last non-empty line.
    answer = next(
        (l.split("Answer:")[-1].strip() for l in text.splitlines() if "Answer:" in l),
        text.splitlines()[-1],
    )
    return {"candidates": [{"reasoning": text, "answer": answer}]}


# Smoke-test: generate one chain manually (costs ~1 API call)
test_state = {"question": "What is 7 * 8?"}
result = generate(test_state)
print("Answer extracted:", result["candidates"][0]["answer"])
print("\nFull reasoning:")
print(result["candidates"][0]["reasoning"])

---
## Part 5 — Process Reward Model: Scoring with an LLM Judge

### The JUDGE_PROMPT

The judge prompt is carefully designed to score *process*, not just *outcome*:

```
Criterion 1: Step clarity   — each step is labeled and easy to follow
Criterion 2: Logical rigor  — no skipped deductions or unjustified leaps
Criterion 3: Correct answer — the final answer is right
```

Output is **structured JSON** with a 1-10 score and a one-sentence critique. JSON output (instead of free text) makes the score easy to parse reliably.

### Why use an LLM as the judge?

A trained PRM (like Cobbe et al.'s verifier) requires labeled step-level data, which is expensive. An LLM judge (sometimes called *LLM-as-a-judge* or *constitutional AI scoring*) provides a reasonable approximation with zero additional training. The judge uses `temperature=0` to make its scoring deterministic — running it twice on the same chain should give the same score.

### Parsing gotcha

LLMs sometimes wrap JSON in markdown code fences (` ```json ... ``` `). The parsing uses `raw.find("{")` and `raw.rfind("}")` to extract just the JSON object, ignoring any surrounding markdown.

In [ ]:
JUDGE_PROMPT = """\
You are a process reward model. Score this reasoning chain on 3 criteria:
  1. Step clarity   — each step is labeled and easy to follow
  2. Logical rigor  — no skipped deductions or unjustified leaps
  3. Correct answer — the final answer is right

Question: {question}

Reasoning chain:
{reasoning}

Respond with ONLY valid JSON (no markdown):
{{"score": <integer 1-10>, "critique": "<one sentence>"}}"""


# Inspect the prompt
sample_q = "What is 2 + 2?"
sample_r = "Step 1: 2 + 2 = 4. Answer: 4"
print(JUDGE_PROMPT.format(question=sample_q, reasoning=sample_r))

In [ ]:
def score_all(state: BonState) -> dict:
    """Score every candidate with the LLM judge; pick highest-scoring chain."""
    scored = []
    for c in state["candidates"]:
        raw = _judge.invoke([
            {"role": "user", "content": JUDGE_PROMPT.format(
                question=state["question"], reasoning=c["reasoning"]
            )}
        ]).content
        # Strip markdown fences if present
        s, e = raw.find("{"), raw.rfind("}") + 1
        try:
            data = json.loads(raw[s:e])
            score, critique = int(data.get("score", 5)), data.get("critique", "")
        except Exception:
            score, critique = 5, ""
        scored.append({**c, "score": score, "critique": critique})

    best = max(scored, key=lambda c: c["score"])
    return {"scored": scored, "best": best["answer"]}


# Demonstrate the JSON parsing logic in isolation
raw_with_fences = '```json\n{"score": 8, "critique": "Clear steps but missed a unit."}\n```'
s, e = raw_with_fences.find("{"), raw_with_fences.rfind("}") + 1
parsed = json.loads(raw_with_fences[s:e])
print("Parsed from fenced output:", parsed)

---
## Part 6 — Graph Assembly

The graph structure is:

```
START
  │  (conditional edge via dispatch)
  ├──► generate   (N parallel invocations via Send)
  ├──► generate
  ├──► generate
  └──► generate
         │
         ▼
       score_all  (runs once all N candidates are collected)
         │
         ▼
        END
```

### Key LangGraph APIs used

| API | Purpose |
|-----|---------|
| `StateGraph(BonState)` | Creates a graph that carries `BonState` |
| `g.add_node(name, fn)` | Registers a node with a processing function |
| `g.add_conditional_edges(START, dispatch, ["generate"])` | Runs `dispatch()` to get routing targets |
| `g.add_edge("generate", "score_all")` | After any `generate` completes, route to `score_all` |
| `g.compile()` | Returns a `CompiledGraph` (the `app`) |
| `app.invoke(initial_state)` | Runs the graph synchronously |

In [ ]:
def create_workflow():
    g = StateGraph(BonState)
    g.add_node("generate",  generate)
    g.add_node("score_all", score_all)
    g.add_conditional_edges(START, dispatch, ["generate"])
    g.add_edge("generate",  "score_all")
    g.add_edge("score_all", END)
    return g.compile()


app = create_workflow()
print("Graph compiled successfully.")
print("Nodes:", list(app.nodes.keys()))

---
## Part 7 — Problems: What We're Testing

The two problems are chosen to exercise different failure modes in LLM reasoning:

### Problem 1 — Rate Problem
> A train travels 120 miles in 90 minutes. How many miles will it travel in 2.5 hours at the same speed?

**Why tricky:** Requires unit conversion (minutes → hours) before applying the rate formula. A chain that skips the conversion and treats 90 minutes as 90 hours will get a wildly wrong answer. A PRM will penalize the missing conversion step even if the arithmetic after the error is perfect.

**Correct answer:** speed = 120/1.5 = 80 mph → 80 × 2.5 = **200 miles**

### Problem 2 — Constraint Logic
> Alice, Bob, and Carol each own exactly one pet: a cat, a dog, or a fish.
> Alice does not own the cat. Bob does not own the fish. Carol does not own the dog.
> Who owns the cat?

**Why tricky:** Requires eliminating possibilities systematically. A chain that makes unjustified leaps ("since Alice can't have the cat, Alice has the dog") without checking consistency will be caught by the logical rigor criterion.

**Correct answer:** Alice → dog or fish, Bob → cat or dog, Carol → cat or fish. Bob must own the cat (only option after eliminating) → **Bob**.

In [ ]:
PROBLEMS = [
    {
        "label": "Rate Problem",
        "question": (
            "A train travels 120 miles in 90 minutes. "
            "How many miles will it travel in 2.5 hours at the same speed?"
        ),
    },
    {
        "label": "Constraint Logic",
        "question": (
            "Alice, Bob, and Carol each own exactly one pet: a cat, a dog, or a fish. "
            "Alice does not own the cat. Bob does not own the fish. Carol does not own the dog. "
            "Who owns the cat?"
        ),
    },
]

print(f"Running {len(PROBLEMS)} problems, {N_SAMPLES} chains each.")
print(f"Total LLM calls: {len(PROBLEMS) * (N_SAMPLES + N_SAMPLES)} (generate + judge per chain)")

---
## Part 8 — End-to-End Run

Now we run the full pipeline. For each problem:
1. `dispatch` fans out 4 `generate` calls in parallel
2. Each `generate` call returns one chain-of-thought candidate
3. `score_all` scores all 4 candidates with the judge
4. Results are sorted from worst to best so the winner appears last

This cell will make approximately **16 API calls** (4 generate + 4 judge per problem × 2 problems). Typical runtime: 30-60 seconds.

In [ ]:
print(f"=== 97 · Best-of-N Sampling | N={N_SAMPLES} | process reward model ===")
print("Ref: Cobbe et al. 2021 — arxiv.org/abs/2110.14168\n")

all_results = []

for problem in PROBLEMS:
    print(f"[{problem['label']}]  {problem['question']}\n")

    result = app.invoke({
        "question":   problem["question"],
        "candidates": [],
        "scored":     [],
        "best":       "",
    })

    all_results.append({"problem": problem, "result": result})

    # Print all scored candidates from worst to best so the winner is last.
    ranked = sorted(result["scored"], key=lambda c: c["score"])
    for i, c in enumerate(ranked, start=1):
        print(f"  Chain {i}  score={c['score']}/10  → {c['answer'][:80]}")
        print(f"          {c['critique']}")
    print()
    print(f"  BEST ANSWER: {result['best']}")
    print("-" * 70)
    print()

print("Takeaway: best-of-N scales inference compute instead of training compute.")
print("A process reward model scores intermediate steps — not just the final answer.")

---
## Part 9 — Analysis: Understanding the Ranked Outputs

Let's programmatically examine the results to understand what differentiated the best chain from the worst.

In [ ]:
# Analyze score distribution across all problems
for entry in all_results:
    problem = entry["problem"]
    result  = entry["result"]
    scored  = result["scored"]

    scores = [c["score"] for c in scored]
    print(f"=== {problem['label']} ===")
    print(f"Scores: {sorted(scores)}")
    print(f"Min: {min(scores)}  Max: {max(scores)}  Spread: {max(scores) - min(scores)}")
    print(f"Best answer: {result['best']}")
    print()

    # Show best and worst chains for comparison
    ranked = sorted(scored, key=lambda c: c["score"])
    worst  = ranked[0]
    best   = ranked[-1]

    print(f"  WORST  (score {worst['score']}/10): {worst['critique']}")
    print(f"  BEST   (score {best['score']}/10):  {best['critique']}")
    print()

In [ ]:
# Show the full winning reasoning chain for the first problem
entry  = all_results[0]
scored = entry["result"]["scored"]
winner = max(scored, key=lambda c: c["score"])

print(f"=== Winning chain for: {entry['problem']['label']} ===")
print(f"Score: {winner['score']}/10")
print(f"Critique: {winner['critique']}")
print()
print("--- Full Reasoning ---")
print(winner["reasoning"])

---
## Part 10 — Deep Dive: Why Temperature Matters

The generation temperature is a crucial hyperparameter for Best-of-N.

```
Temperature → 0 (deterministic)
  All N chains are nearly identical → no benefit from multiple samples

Temperature → 1+ (high entropy)
  Chains are maximally diverse but may be incoherent

Temperature ≈ 0.6–0.9 (sweet spot)
  Diverse enough that at least one chain reasons correctly
  Coherent enough that the judge can meaningfully compare them
```

Let's demonstrate the effect of temperature on diversity:

In [ ]:
# Compare generation diversity at two temperature settings
# (makes 4 API calls — 2 at low temp, 2 at high temp)

q = "What is the square root of 144?"

_low_temp  = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
_high_temp = ChatOpenAI(model="gpt-4o-mini", temperature=1.0)

def quick_generate(llm, question):
    reply = llm.invoke([
        {"role": "system", "content": "Solve step by step."},
        {"role": "user",   "content": question},
    ])
    return reply.content.strip()

print("LOW TEMPERATURE (0.1) — expect near-identical chains:")
for i in range(2):
    chain = quick_generate(_low_temp, q)
    print(f"  Chain {i+1}: {chain[:120]}...")
    print()

print("HIGH TEMPERATURE (1.0) — expect diverse chains:")
for i in range(2):
    chain = quick_generate(_high_temp, q)
    print(f"  Chain {i+1}: {chain[:120]}...")
    print()

---
## Part 11 — Scaling: How N Affects Result Quality

Cobbe et al. showed that Best-of-N accuracy follows a predictable curve as N increases:

```
Accuracy
  │         ←── Best-of-N with strong verifier
  │       ╱
  │     ╱
  │   ╱
  │ ╱
  │╱___________  ←── Best-of-1 (standard generation)
  └──────────────────────────────────────────── N
    1  2  4  8  16  32  64  128
```

The gains are largest going from N=1 → N=4 → N=16. After N=64, returns diminish. The curve shape depends heavily on how good the verifier is — a weak verifier flattens the curve.

Let's look at how the best score changes as we vary N using our already-collected data:

In [ ]:
# Simulate "what would best-of-k look like" for k = 1, 2, 3, 4
# using the 4 candidates we already generated
import random

for entry in all_results:
    problem = entry["problem"]
    scored  = entry["result"]["scored"]
    all_scores = [c["score"] for c in scored]

    print(f"=== {problem['label']} — Simulated Best-of-K ===")
    print(f"All scores available: {sorted(all_scores)}")

    # For each k, compute best score achievable from k random draws
    random.seed(42)
    for k in [1, 2, 3, 4]:
        sample = random.sample(all_scores, k)
        best_k = max(sample)
        print(f"  Best-of-{k} (sample={sample}): best score = {best_k}/10")
    print()

---
## Part 12 — Graph Visualization

LangGraph can draw the compiled graph structure. Let's inspect it:

In [ ]:
# Print the graph structure as a text representation
print("Graph nodes:", list(app.nodes.keys()))
print()

# Try to get the Mermaid diagram
try:
    mermaid = app.get_graph().draw_mermaid()
    print("Mermaid diagram:")
    print(mermaid)
except Exception as exc:
    # Fallback: print the ASCII representation
    print("ASCII graph:")
    print("""
    START
      │  dispatch() → [Send(generate), Send(generate), Send(generate), Send(generate)]
      │
      ├──► generate (chain 0)
      ├──► generate (chain 1)  ← all run in parallel
      ├──► generate (chain 2)
      └──► generate (chain 3)
               │
               ▼  (candidates list merged via operator.add)
           score_all
               │
               ▼
             END
    """)
    print(f"(draw_mermaid error: {exc})")

---
## Part 13 — The Full Pipeline in One Function (from main.py)

For reference, here is the exact `main()` function from the source repository. This is what `python main.py` runs:

In [ ]:
def main() -> None:
    print(f"=== 97 · Best-of-N Sampling | N={N_SAMPLES} | process reward model ===")
    print("Ref: Cobbe et al. 2021 — arxiv.org/abs/2110.14168\n")

    workflow = create_workflow()

    for problem in PROBLEMS:
        print(f"[{problem['label']}]  {problem['question']}\n")

        result = workflow.invoke({
            "question":   problem["question"],
            "candidates": [],
            "scored":     [],
            "best":       "",
        })

        # Print all scored candidates from worst to best so the winner is last.
        ranked = sorted(result["scored"], key=lambda c: c["score"])
        for i, c in enumerate(ranked, start=1):
            print(f"  Chain {i}  score={c['score']}/10  → {c['answer'][:80]}")
            print(f"          {c['critique']}")
        print()
        print(f"  BEST ANSWER: {result['best']}")
        print("-" * 70)
        print()

    print("Takeaway: best-of-N scales inference compute instead of training compute.")
    print("A process reward model scores intermediate steps — not just the final answer.")


# Uncomment to re-run the full main function:
# main()
print("main() defined — uncomment the last line to run.")

---
## Part 14 — Tradeoffs and When to Use Best-of-N

### When Best-of-N works well

| Situation | Why it helps |
|-----------|-------------|
| Reasoning tasks with variable quality | High temp → diverse paths; PRM picks the rigorous one |
| No fine-tuning budget | Inference-time compute vs. training compute |
| Tasks with a good verifier | Math, logic, code (where correctness is checkable) |
| Latency-tolerant applications | N parallel calls adds latency but improves quality |

### When Best-of-N is a poor fit

| Situation | Why it fails |
|-----------|-------------|
| No reliable verifier | Can't pick the best if the judge is weak |
| Latency-critical (< 1s budget) | N parallel calls still multiplies base latency |
| Open-ended creative tasks | No "correct" answer to reward |
| Cost-constrained | N calls = N × cost per call |

### Alternatives

| Technique | Tradeoff |
|-----------|----------|
| Self-consistency (majority vote) | No verifier needed, but weaker selection |
| MCTS (Monte Carlo Tree Search) | Better search, much more complex to implement |
| Fine-tuning on verifier signal | Better long-term, requires labeled data |
| Beam search | Guided generation, no post-hoc scoring |

---
## Exercises

### Exercise 1 — Add a Third Problem

Add a new problem to `PROBLEMS` that tests a different reasoning skill (e.g., probability, geometry, or syllogistic reasoning). Run the workflow on your new problem and inspect which chain wins and why.

```python
# Hint: add a dict with "label" and "question" keys to PROBLEMS
# Then call app.invoke({"question": your_question, "candidates": [], "scored": [], "best": ""})
```

---

### Exercise 2 — Modify the Judge Criteria

Change `JUDGE_PROMPT` to score on **different criteria** — for example:
- Conciseness (penalize verbose chains)
- Mathematical notation quality
- Confidence calibration

Run the Rate Problem with your modified judge. Does the winner change?

---

### Exercise 3 — Deterministic Majority Vote Baseline

Implement a `majority_vote(scored)` function that ignores the PRM scores and instead picks the most common `answer` string among the candidates. Compare its choice to the PRM winner.

```python
from collections import Counter

def majority_vote(candidates: list[dict]) -> str:
    # your implementation here
    pass
```

In [ ]:
# ============================================================
# ANSWER KEY — Exercise 1: Add a Third Problem
# ============================================================

NEW_PROBLEM = {
    "label": "Probability",
    "question": (
        "A bag contains 3 red marbles and 7 blue marbles. "
        "You draw two marbles without replacement. "
        "What is the probability that both are red?"
    ),
}

print(f"[{NEW_PROBLEM['label']}]  {NEW_PROBLEM['question']}\n")

result = app.invoke({
    "question":   NEW_PROBLEM["question"],
    "candidates": [],
    "scored":     [],
    "best":       "",
})

ranked = sorted(result["scored"], key=lambda c: c["score"])
for i, c in enumerate(ranked, start=1):
    print(f"  Chain {i}  score={c['score']}/10  → {c['answer'][:80]}")
    print(f"          {c['critique']}")
print()
print(f"  BEST ANSWER: {result['best']}")
# Expected: 3/10 * 2/9 = 6/90 = 1/15

In [ ]:
# ============================================================
# ANSWER KEY — Exercise 2: Modified Judge Criteria
# ============================================================

CONCISE_JUDGE_PROMPT = """\
You are a process reward model that values conciseness and correctness.
Score this reasoning chain on 3 criteria:
  1. Conciseness    — minimum steps needed, no unnecessary text
  2. Logical rigor  — no skipped deductions or unjustified leaps
  3. Correct answer — the final answer is right

Question: {question}

Reasoning chain:
{reasoning}

Respond with ONLY valid JSON (no markdown):
{{"score": <integer 1-10>, "critique": "<one sentence>"}}"""


def score_all_concise(state: BonState) -> dict:
    """Judge variant that rewards concise reasoning."""
    scored = []
    for c in state["candidates"]:
        raw = _judge.invoke([
            {"role": "user", "content": CONCISE_JUDGE_PROMPT.format(
                question=state["question"], reasoning=c["reasoning"]
            )}
        ]).content
        s, e = raw.find("{"), raw.rfind("}") + 1
        try:
            data = json.loads(raw[s:e])
            score, critique = int(data.get("score", 5)), data.get("critique", "")
        except Exception:
            score, critique = 5, ""
        scored.append({**c, "score": score, "critique": critique})

    best = max(scored, key=lambda c: c["score"])
    return {"scored": scored, "best": best["answer"]}


# Build a graph variant with the concise judge
def create_concise_workflow():
    g = StateGraph(BonState)
    g.add_node("generate",  generate)
    g.add_node("score_all", score_all_concise)  # swapped
    g.add_conditional_edges(START, dispatch, ["generate"])
    g.add_edge("generate",  "score_all")
    g.add_edge("score_all", END)
    return g.compile()


app_concise = create_concise_workflow()
rate_problem = PROBLEMS[0]

print("Running Rate Problem with CONCISE judge...\n")
result_concise = app_concise.invoke({
    "question":   rate_problem["question"],
    "candidates": [],
    "scored":     [],
    "best":       "",
})

ranked_concise = sorted(result_concise["scored"], key=lambda c: c["score"])
for i, c in enumerate(ranked_concise, start=1):
    print(f"  Chain {i}  score={c['score']}/10  → {c['answer'][:60]}")
    print(f"          {c['critique']}")
print(f"\n  BEST (concise judge): {result_concise['best']}")

In [ ]:
# ============================================================
# ANSWER KEY — Exercise 3: Majority Vote Baseline
# ============================================================

from collections import Counter


def majority_vote(candidates: list[dict]) -> str:
    """Pick the most common answer string, ignoring PRM scores."""
    # Normalize: strip whitespace and lowercase for comparison
    normalized = [c["answer"].strip().lower() for c in candidates]
    counter = Counter(normalized)
    most_common_normalized, count = counter.most_common(1)[0]
    # Return the original-case version of the most common answer
    for c in candidates:
        if c["answer"].strip().lower() == most_common_normalized:
            return c["answer"]
    return candidates[0]["answer"]


print("Comparing PRM winner vs Majority Vote for each problem:\n")

for entry in all_results:
    problem  = entry["problem"]
    result   = entry["result"]
    scored   = result["scored"]

    prm_winner      = result["best"]
    majority_winner = majority_vote(scored)

    print(f"[{problem['label']}]")
    print(f"  All answers:      {[c['answer'][:40] for c in scored]}")
    print(f"  PRM winner:       {prm_winner[:60]}")
    print(f"  Majority winner:  {majority_winner[:60]}")
    print(f"  Same decision:    {prm_winner.strip().lower() == majority_winner.strip().lower()}")
    print()

---
## Workshop Complete

### What you built

A **Best-of-N sampling pipeline** using LangGraph's `Send()` fan-out pattern and an LLM-as-PRM judge that scores reasoning quality rather than just final answers.

### Key takeaways

1. **Inference-time scaling** is a practical alternative to fine-tuning — spend more tokens at inference, not more GPU-hours at training.
2. **Process reward models** are more robust than outcome-only scoring — they catch valid-looking answers backed by broken reasoning.
3. **`operator.add` on Annotated list fields** is the LangGraph idiom for safely collecting results from parallel nodes.
4. **Temperature controls diversity** — use high temperature for generation (diversity) and temperature=0 for judging (stability).
5. **LLM-as-judge** is a practical PRM substitute when labeled step data isn't available.

### Next: example 98 — Reflexion

Reflexion extends Best-of-N by making the model *learn from its failures* within a single inference session: generate → critique → reflect → regenerate, using verbal reinforcement rather than weight updates.

---
*Reference: Cobbe et al. 2021 — Training Verifiers to Solve Math Word Problems — arxiv.org/abs/2110.14168*